In [1]:
# Only accelerate is needed — rest is pre-installed in Colab
!pip install accelerate anthropic -q
print('Done!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 478.3/478.3 kB 16.4 MB/s eta 0:00:00
Done!


In [2]:
import torch
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Name:', torch.cuda.get_device_name(0))
else:
    print('Running on CPU — go to Runtime > Change runtime type > T4 GPU')

GPU Available: True
GPU Name: Tesla T4


## LOADING DATASET

In [6]:
# Cell 3 — Upload and Load Kaggle Dataset
from google.colab import files
import pandas as pd

print('choose train.csv file from your system...')
uploaded = files.upload()

# Load CSV
df = pd.read_csv('train.csv')
print(f'Dataset loaded! Total rows: {df.shape[0]}')
print(f'Columns: {df.columns.tolist()}')
print(df.head(3))

Apni train.csv file select karo...


Saving train.csv to train.csv
Dataset loaded! Total rows: 159571
Columns: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   
2  000113f07ec002fd  Hey man, I'm really not trying to edit war. It...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             0        0       0       0              0  
1             0        0       0       0              0  
2             0        0       0       0              0  


## LABELING OF DATASET

In [9]:
# Preparing labels
texts = df['comment_text'].tolist()

# In this there are multiple type of bias available like--
# toxic, severe_toxic, obscene, threat, insult, identity_hate

labels = []
for _, row in df.iterrows():
    # For bias value is 1....
    if row['toxic'] == 1 or row['insult'] == 1 or row['identity_hate'] == 1 or row['threat'] == 1:
        labels.append(1)
    else:
        labels.append(0)

# Choose 5000
texts  = texts[:5000]
labels = labels[:5000]

print(f'Total samples:        {len(texts)}')
print(f'Biased   (label=1):   {sum(labels)}')
print(f'Fair     (label=0):   {len(labels) - sum(labels)}')

Total samples:        5000
Biased   (label=1):   525
Fair     (label=0):   4475


## USING DISTILBERT (TRANSFORMER)

In [10]:
from transformers import DistilBertTokenizer

print('Loading tokenizer...')
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

encodings = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors='pt'
)

print('Tokenization done!')
print(f'Input IDs shape: {encodings["input_ids"].shape}')

Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenization done!
Input IDs shape: torch.Size([5000, 128])


## TRAIN/VAL SPLIT

In [11]:
class BiasDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

split = int(0.8 * len(texts))

train_dataset = BiasDataset(
    {k: v[:split] for k, v in encodings.items()},
    labels[:split]
)
val_dataset = BiasDataset(
    {k: v[split:] for k, v in encodings.items()},
    labels[split:]
)

print(f'Train samples: {len(train_dataset)}')
print(f'Val samples:   {len(val_dataset)}')

Train samples: 4000
Val samples:   1000


## MODEL TRAINING

In [16]:
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)
model.to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()
    return {'accuracy': accuracy}

training_args = TrainingArguments(
    output_dir='./model',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print('Starting training...')
trainer.train()
print('Training complete!')

Training on: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.144378,0.119252,0.952000
2,0.097880,0.182664,0.958000
3,0.039206,0.228473,0.959000
4,0.003107,0.279951,0.957000
5,0.000535,0.317324,0.948000
6,0.001827,0.306211,0.959000
7,0.000092,0.308266,0.959000
8,0.000069,0.317351,0.959000
9,0.000057,0.322476,0.957000
10,0.000052,0.327645,0.957000


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Training complete!


## MODEL SAVE

In [17]:
import os
os.makedirs('./model/final', exist_ok=True)
model.save_pretrained('./model/final')
tokenizer.save_pretrained('./model/final')
print('Model saved to ./model/final')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./model/final


## MODEL TESTING

In [19]:
# Testing the model

model.eval()

test_sentences = [
    # BIASED TEST DATA
    'You stupid idiot, go back to your country where you belong.',
    'Women are too emotional to be good leaders, they should stay home.',
    'All Muslims are terrorists and should be banned from this country.',
    'Black people are criminals and should not be trusted.',
    'Kill yourself, nobody wants you here you worthless piece of garbage.',

    # UNBIASED TEST DATA
    'We are looking for an experienced professional for this role.',
    'The weather today is sunny and pleasant.',
    'Please submit your assignment before the deadline.',
]

print('--- Model Test Results ---\n')
for sent in test_sentences:
    enc = tokenizer(sent, return_tensors='pt', truncation=True, padding=True).to(device)
    with torch.no_grad():
        out = model(**enc)
    prob = torch.softmax(out.logits, dim=1)
    score = prob[0][1].item()
    label = '🔴 BIASED/TOXIC' if score > 0.5 else '🟢 NOT BIASED'
    print(f'Text:   {sent[:70]}')
    print(f'Result: {label} | Score: {score:.2%}\n')

--- Model Test Results ---

Text:   You stupid idiot, go back to your country where you belong.
Result: 🔴 BIASED/TOXIC | Score: 97.46%

Text:   Women are too emotional to be good leaders, they should stay home.
Result: 🟢 NOT BIASED | Score: 6.00%

Text:   All Muslims are terrorists and should be banned from this country.
Result: 🔴 BIASED/TOXIC | Score: 89.77%

Text:   Black people are criminals and should not be trusted.
Result: 🔴 BIASED/TOXIC | Score: 89.01%

Text:   Kill yourself, nobody wants you here you worthless piece of garbage.
Result: 🔴 BIASED/TOXIC | Score: 96.95%

Text:   We are looking for an experienced professional for this role.
Result: 🟢 NOT BIASED | Score: 0.73%

Text:   The weather today is sunny and pleasant.
Result: 🟢 NOT BIASED | Score: 1.14%

Text:   Please submit your assignment before the deadline.
Result: 🟢 NOT BIASED | Score: 0.42%



## MODEL ACCURACY REPORT ON VAL DATA

In [22]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

model.eval()
all_preds = []
all_labels = []

print('Running predictions on validation set...')
for i in range(0, len(val_dataset), 32):
    batch      = [val_dataset[j] for j in range(i, min(i+32, len(val_dataset)))]
    input_ids      = torch.stack([b['input_ids']      for b in batch]).to(device)
    attention_mask = torch.stack([b['attention_mask'] for b in batch]).to(device)
    batch_labels   = [b['labels'].item()              for b in batch]

    with torch.no_grad():
        out = model(input_ids=input_ids, attention_mask=attention_mask)

    all_preds.extend(torch.argmax(out.logits, dim=1).cpu().tolist())
    all_labels.extend(batch_labels)

acc = accuracy_score(all_labels, all_preds)

print('\n' + '='*50)
print('        MODEL ACCURACY REPORT')
print('='*50)
print(f'  Overall Accuracy : {acc*100:.2f}%')
print('='*50)
print(classification_report(all_labels, all_preds, target_names=['Fair (0)', 'Biased (1)']))

cm = confusion_matrix(all_labels, all_preds)
print('Confusion Matrix:')
print(f'                 Predicted Fair  Predicted Biased')
print(f'Actual Fair      {cm[0][0]:^14}  {cm[0][1]:^15}')
print(f'Actual Biased    {cm[1][0]:^14}  {cm[1][1]:^15}')

filled = int(acc * 40)
print(f'\n[{"█" * filled}{"░" * (40-filled)}] {acc*100:.2f}%')

Running predictions on validation set...

        MODEL ACCURACY REPORT
  Overall Accuracy : 95.20%
              precision    recall  f1-score   support

    Fair (0)       0.98      0.96      0.97       894
  Biased (1)       0.74      0.85      0.79       106

    accuracy                           0.95      1000
   macro avg       0.86      0.91      0.88      1000
weighted avg       0.96      0.95      0.95      1000

Confusion Matrix:
                 Predicted Fair  Predicted Biased
Actual Fair           862              32       
Actual Biased          16              90       

[██████████████████████████████████████░░] 95.20%


## Model Accuracy Report on TEST DATA

In [26]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

# Last rows for testing
# the data that did not use for training (we chose first 5000 only)
from google.colab import files
import pandas as pd


df_full = pd.read_csv('train.csv')

# Last 1000 rows from training
df_test_portion = df_full.iloc[5000:6000]

test_texts  = df_test_portion['comment_text'].tolist()
test_labels = []
for _, row in df_test_portion.iterrows():
    if row['toxic']==1 or row['insult']==1 or row['identity_hate']==1 or row['threat']==1:
        test_labels.append(1)
    else:
        test_labels.append(0)

print(f'Test samples : {len(test_texts)}')
print(f'Biased       : {sum(test_labels)}')
print(f'Fair         : {len(test_labels) - sum(test_labels)}')

# ── Tokenize test data ────────────────────────────────────────
test_enc = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors='pt'
)

test_dataset = BiasDataset(test_enc, test_labels)

# ── Run Predictions ───────────────────────────────────────────
model.eval()
all_preds  = []
all_labels = []

print('\nRunning predictions...')
for i in range(0, len(test_dataset), 32):
    batch          = [test_dataset[j] for j in range(i, min(i+32, len(test_dataset)))]
    input_ids      = torch.stack([b['input_ids']      for b in batch]).to(device)
    attention_mask = torch.stack([b['attention_mask'] for b in batch]).to(device)
    batch_labels   = [b['labels'].item()              for b in batch]

    with torch.no_grad():
        out = model(input_ids=input_ids, attention_mask=attention_mask)

    all_preds.extend(torch.argmax(out.logits, dim=1).cpu().tolist())
    all_labels.extend(batch_labels)

# ── Accuracy Report ───────────────────────────────────────────
acc = accuracy_score(all_labels, all_preds)

print('\n' + '='*50)
print('      TEST SET ACCURACY REPORT')
print('='*50)
print(f'  Total Test Samples : {len(test_labels)}')
print(f'  Overall Accuracy   : {acc*100:.2f}%')
print('='*50)
print(classification_report(
    all_labels, all_preds,
    target_names=['Fair (0)', 'Biased (1)']
))

cm = confusion_matrix(all_labels, all_preds)
print('Confusion Matrix:')
print(f'                 Predicted Fair  Predicted Biased')
print(f'Actual Fair      {cm[0][0]:^14}  {cm[0][1]:^15}')
print(f'Actual Biased    {cm[1][0]:^14}  {cm[1][1]:^15}')

filled = int(acc * 40)
print(f'\n[{"█" * filled}{"░" * (40-filled)}] {acc*100:.2f}%')

Test samples : 1000
Biased       : 106
Fair         : 894

Running predictions...

      TEST SET ACCURACY REPORT
  Total Test Samples : 1000
  Overall Accuracy   : 95.80%
              precision    recall  f1-score   support

    Fair (0)       0.98      0.97      0.98       894
  Biased (1)       0.79      0.83      0.81       106

    accuracy                           0.96      1000
   macro avg       0.88      0.90      0.89      1000
weighted avg       0.96      0.96      0.96      1000

Confusion Matrix:
                 Predicted Fair  Predicted Biased
Actual Fair           870              24       
Actual Biased          18              88       

[██████████████████████████████████████░░] 95.80%
